In [1]:
import os
import time
import functools
from openai import OpenAI
from dotenv import load_dotenv
import traceback
import requests
import json

def retry(max_retries: int = 5, sleep_seconds: float = 15.0):
    """
    重试装饰器：函数报错时等待 sleep_seconds 秒后重试，最多重试 max_retries 次。
    超过重试上限后，抛出最后一次的异常。
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_retries + 2):  # 1次正常 + max_retries次重试
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    if attempt <= max_retries:
                        print(f"[retry] {func.__name__} 第{attempt}次失败: {traceback.format_exc()}，{sleep_seconds}秒后重试...")
                        time.sleep(sleep_seconds)
                    else:
                        print(f"[retry] {func.__name__} 已达最大重试次数({max_retries})，放弃。")
            raise last_exc
        return wrapper
    return decorator


@retry(max_retries=3, sleep_seconds=5.0)
def qwen(prompt, web_search: bool = False, enable_thinking: bool = False):
    client = OpenAI(
        # 从环境变量读取，或直接填写 api_key="sk-xxx"
        api_key="sk-fb07e345e2b04562ad5acf2d4bfee8fa", #os.getenv("QWEN_API_KEY"),
        # 北京地域（默认）
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"

    )

    completion = client.chat.completions.create(
        model=os.getenv("INFER_LLM_NAME", "qwen3-max"),
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        # stream=True,  # 如需流式输出，取消注释
        extra_body={
            "enable_search": web_search,  # 开启联网搜索
            "enable_thinking": enable_thinking,  # 开关思考功能
            # "thinking_budget": 81920   # 可选：限制思考过程最大Token数（默认32768）
            # "search_options": {
            #     "forced_search": True,        # 强制开启搜索（默认false，由模型判断是否使用）
            #     "search_strategy": "turbo",   # 搜索策略：turbo(默认)/max/agent/agent_max
            #     "enable_source": True,        # 返回搜索结果来源（仅部分模型支持）
            #     "enable_citation": True,      # 开启角标标注[1]样式（需enable_source为true）
            #     "citation_format": "[ref_<number>]"  # 角标样式：[<number>] 或 [ref_<number>]
            # }
        }

    )
    if enable_thinking:
        res ={
            "content": completion.choices[0].message.content,
            "reasoning_content":completion.choices[0].message.reasoning_content
        }
    else:
        res ={
            "content": completion.choices[0].message.content,
            "reasoning_content":""
        }

    return res

In [2]:
#"sk-5a5c8fea7cc14d779a201d8ab0be8f91"

@retry(max_retries=3, sleep_seconds=5.0)
def chat(messages: str, vendor: str = "aliyun", model: str = "qwen2.5-32b-instruct") -> str:
    URL = f"http://mlp.paas.dc.servyou-it.com/mudgate/api/llm/{vendor}/v1/chat/completions"
    app_id = "sk-0609aa6d08de4413a72e14b3fb8fbab1"
    HEADERS = {"Content-Type": "application/json", "Authorization": app_id}
    messages = [{"role": "user", "content": messages}]
    PAYLOAD = {
        "appId": app_id,
        "model": model,
        "messages": messages,
        "stream": False,
        "top_p": 0.7,
        "temperature": 0.5,
        # "enable_thinking": True,
        # "enable_search":True
    }
    response = requests.post(URL, data=json.dumps(PAYLOAD), headers=HEADERS).json()
    if "success" in response:
        raise Exception(response["errorContext"])
    return response["choices"][0]["message"]#["content"]

In [3]:
chat("你是谁")

{'role': 'assistant',
 'content': '我是Qwen，由阿里云开发的大型语言模型。我被设计用来帮助人们解答问题、提供信息和进行各种对话。有什么我可以帮助你的吗？'}

# 读数据

In [4]:
import pandas as pd

In [5]:
ans_data = pd.read_excel("./output_kh_zqrz317_experts.xlsx", engine = "openpyxl")[['question','answer']]#.fillna("", inplace = True)

In [6]:
data = pd.read_excel("./zqrz_fjrjg_data.xlsx", engine = "openpyxl")#.fillna("", inplace = True)
data.columns
data.rename(columns = {"序号":"id","问题":"question","答案":"answer"}, inplace = True)
data['policy'] = None
data['reasoning'] = None

In [7]:
data = ans_data.merge(data, how = "inner", on = "question")

In [8]:
data.head()

,question,answer,QID,confirmed,reasoning,judgement,policy
0,作为一家贷款公司，向A公司借款100万元到期后仅收回88万元导致12万元损失，该损失在税务申...,作为专业贷款机构，您应按照税收政策计提贷款损失准备金，其税前扣除有其特殊规定。这笔12万元的...,Q47,非金融机构借贷业务,None,True,None
1,G100000《关联业务往来汇总表》“201年度平均关联债权投资金额”栏如何填写,<p>《关联业务往来汇总表》“201年度平均关联债权投资金额”栏＝表G107000第N行第8...,Q53,非金融机构借贷业务,None,True,None
2,同一老板控制的两家企业，发生资金借贷，是否需要进行关联申报,<p>两家企业同为第三方持股达到25%以上的，构成关联关系；资金借贷是关联交易类型中的资金融...,Q54,非金融机构借贷业务,None,True,None
3,（1）集团100%控股我们公司，集团收走我司2000万资金做资金统筹，资金后期会还回来，公司...,（1）2026 年 1 月 1 日前，若符合集团内单位无偿借贷条件，可享受免征增值税优惠；2...,Q56,非金融机构借贷业务,None,True,None
4,统借统还税务相关问题,一、 统借统还业务定义\n统借统还业务是指：企业集团或者企业集团中的核心企业向金融机构借款或...,Q57,非金融机构借贷业务,None,True,None


In [9]:
data.to_csv("./fjrjg_for_hkl.csv")

In [10]:
data['policy'].fillna("暂无", inplace = True)
data['reasoning'].fillna("暂无", inplace = True)

/tmp/ipykernel_1181658/1966434329.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['policy'].fillna("暂无", inplace = True)
/tmp/ipykernel_1181658/1966434329.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

In [11]:
data['answer'] = data['answer'] + "\n\n" + "- 政策依据"  +"\n"+ data['policy']

In [12]:
data.head()

,question,answer,QID,confirmed,reasoning,judgement,policy
0,作为一家贷款公司，向A公司借款100万元到期后仅收回88万元导致12万元损失，该损失在税务申...,作为专业贷款机构，您应按照税收政策计提贷款损失准备金，其税前扣除有其特殊规定。这笔12万元的...,Q47,非金融机构借贷业务,暂无,True,暂无
1,G100000《关联业务往来汇总表》“201年度平均关联债权投资金额”栏如何填写,<p>《关联业务往来汇总表》“201年度平均关联债权投资金额”栏＝表G107000第N行第8...,Q53,非金融机构借贷业务,暂无,True,暂无
2,同一老板控制的两家企业，发生资金借贷，是否需要进行关联申报,<p>两家企业同为第三方持股达到25%以上的，构成关联关系；资金借贷是关联交易类型中的资金融...,Q54,非金融机构借贷业务,暂无,True,暂无
3,（1）集团100%控股我们公司，集团收走我司2000万资金做资金统筹，资金后期会还回来，公司...,（1）2026 年 1 月 1 日前，若符合集团内单位无偿借贷条件，可享受免征增值税优惠；2...,Q56,非金融机构借贷业务,暂无,True,暂无
4,统借统还税务相关问题,一、 统借统还业务定义\n统借统还业务是指：企业集团或者企业集团中的核心企业向金融机构借款或...,Q57,非金融机构借贷业务,暂无,True,暂无


In [13]:
# 拆除头20条进行know-how迭代
cut = int(len(data) * 1)
print(cut)
data_train = data.loc[:cut,:]
test_cut = int(cut*0.7)
data_test = data.loc[test_cut:,:]

152


# 推理迭代

In [14]:
import json5
from tqdm import tqdm
from prompts import single_v0, single_v1, compression_v0, infer_v0, summary_v0, compression_v1, compression_v2, infer_v1, potential_pitfalls

# 一级提炼

In [14]:
import time
import json
import json5
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import os

file_lock = Lock()

def process_single_item_with_retry(c, data_train, eb, chat_func, v0_func, output_file, max_retries=None):
    """
    处理单个数据项，带重试机制直到成功
    
    Args:
        max_retries: 最大重试次数，None表示无限重试直到成功（建议设置上限如100防止死循环）
    """
    q = data_train['question'][c]
    r = data_train['reasoning'][c]
    a = data_train['answer'][c]
    
    retry_count = 0
    base_wait = 3  # 基础等待秒数
    
    while True:
        # 检查是否达到最大重试次数（如果设置了的话）
        if max_retries is not None and retry_count >= max_retries:
            error_info = {
                "index": c,
                "status": "failed",
                "error": "达到最大重试次数",
                "retry_count": retry_count,
                "last_error": last_error_msg
            }
            # 带锁写入失败状态
            with file_lock:
                _update_json_file(output_file, str(c), error_info)
            print(f"[Failed] 第 {c} 项在重试 {max_retries} 次后放弃")
            return c, "failed", None
        
        try:
            if retry_count > 0:
                print(f"[Retry {retry_count}] 第 {c} 项第 {retry_count + 1} 次尝试...")
            
            # 调用模型
            response = chat_func(v0_func(eb, q, r, a))

            # 解析JSON（这里可能会因为模型格式错误而失败）
            try:
                content = json5.loads(response['content'])
            except (json.JSONDecodeError, KeyError, TypeError) as json_err:
                raise Exception(f"JSON解析失败: {str(json_err)} | 原始内容: {str(response.get('content', 'N/A'))[:100]}")
            
            # 验证必需字段
            if 'Know_How' not in content:
                raise Exception(f"响应缺少必需字段 'Know_How'")
            
            know_how = content['Know_How']
            logic_diagnosis = content.get('Logic_Diagnosis', '')
            
            # 构建成功结果
            result = {
                "index": c,
                "Know_How": know_how,
                "Logic_Diagnosis": logic_diagnosis,
                "status": "success",
                "retry_count": retry_count  # 记录实际重试次数
            }
            
            # 带锁写入文件
            with file_lock:
                _update_json_file(output_file, str(c), result)
            
            status_msg = f"第 {c}/{len(data_train)} 项完成"
            if retry_count > 0:
                status_msg += f"（历经 {retry_count} 次重试）"
            print(f"[Success] {status_msg}")
            
            return c, "success", know_how
            
        except Exception as e:
            retry_count += 1
            last_error_msg = str(e)
            
            # 每5次重试打印一次详细错误，避免日志刷屏
            if retry_count % 5 == 1:
                print(f"[Error] 第 {c} 项第 {retry_count} 次失败: {last_error_msg[:150]}...")
                print(traceback.format_exc())
            else:
                print(f"[Error] 第 {c} 项第 {retry_count} 次失败: {last_error_msg[:100]}...")
            
            # 指数退避：等待时间随重试次数指数增长，但最多60秒
            wait_time = 3 # min(base_wait * (2 ** (retry_count - 1)), 60)
            print(f"[Wait] 等待 {wait_time} 秒后重试...")
            time.sleep(wait_time)

def _update_json_file(file_path, key, value):
    """辅助函数：线程安全地更新JSON文件"""
    data_dict = {}
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data_dict = json.load(f)
        except (json.JSONDecodeError, IOError):
            data_dict = {}
    
    data_dict[key] = value
    
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_dict, f, ensure_ascii=False, indent=2)

def kh_extraction_multithread(data_train, max_workers=5, output_file="./kh_results_zqrz317_fjrjg.json", max_retries_per_item=100):
    """
    多线程 Know-How 提取（带无限重试机制）
    
    Args:
        max_workers: 并发线程数（根据API限流调整）
        output_file: 输出JSON文件（以行号为key）
        max_retries_per_item: 每个项目的最大重试次数，None表示真正无限重试（谨慎使用），默认100次
    """
    eb = "债权融资"
    total = len(data_train)
    
    print(f"开始多线程处理（带重试），总数据量: {total}, 并发数: {max_workers}")
    if max_retries_per_item is None:
        print("⚠️  警告：设置为无限重试模式，遇到顽固错误时线程将永不退出")
    else:
        print(f"最大重试次数: {max_retries_per_item} 次/项")
    
    # 初始化输出文件（支持断点续传）
    existing_data = {}
    if os.path.exists(output_file):
        try:
            with open(output_file, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
            print(f"📂 发现已有进度文件，包含 {len(existing_data)} 条记录，自动续传")
        except:
            existing_data = {}
    
    # 筛选未完成的任务（支持断点续传）
    pending_indices = [
        c for c in range(total) 
        if str(c) not in existing_data or existing_data.get(str(c), {}).get('status') != 'success'
    ]
    
    completed = total - len(pending_indices)
    print(f"已完成: {completed}, 待处理: {len(pending_indices)}")
    
    # 主循环
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(
                process_single_item_with_retry, 
                c, data_train, eb, chat, single_v1, output_file, max_retries_per_item
            ): c for c in pending_indices
        }
        
        processed = 0
        for future in as_completed(future_to_index):
            c = future_to_index[future]
            processed += 1
            
            try:
                index, status, _ = future.result()
                if status == "success":
                    completed += 1
                    print(f"✅ 进度: {completed}/{total} ({completed/total*100:.1f}%)")
                else:
                    print(f"❌ 第 {index} 项最终失败")
            except Exception as e:
                print(f"💥 第 {c} 项处理异常: {e}")
            
            # 每完成10%打印一次汇总
            if processed % max(1, len(pending_indices) // 10) == 0:
                print(f"📊 已处理 {processed}/{len(pending_indices)} 个任务")
    
    print(f"\n🏁 全部完成！结果保存于: {output_file}")

# 使用示例：
# 严格无限重试（直到成功，慎用）
# kh_extraction_multithread(data_train, max_workers=5, max_retries_per_item=None)

# 推荐：最多重试100次（防止极端情况死循环）
kh_extraction_multithread(data_train, max_workers=16, max_retries_per_item=100)

开始多线程处理（带重试），总数据量: 152, 并发数: 16
最大重试次数: 100 次/项
📂 发现已有进度文件，包含 152 条记录，自动续传
已完成: 152, 待处理: 0

🏁 全部完成！结果保存于: ./kh_results_zqrz317_fjrjg.json


# 二级提炼：对问题中提取的一级know-how进行整合（若已有业务范围框定，则按业务范围独立整合）

In [15]:
import json
import json5
import time
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import os
from prompts import compression_v0, safe_parse_json

# ── 1. 加载一级提炼结果 ──────────────────────────────────────────────────────
with open("./kh_results_zqrz317_fjrjg.json", "r", encoding="utf-8") as f:
    kh_data = json.load(f)

# 只保留成功且 Know_How 非空的条目，按 index 排序
valid_items = sorted(
    [
        {"index": v["index"], "Know_How": v["Know_How"]}
        for v in kh_data.values()
        if v.get("status") == "success" and v.get("Know_How", "").strip()
    ],
    key=lambda x: x["index"]
)
print(f"有效 Know_How 总数: {len(valid_items)}")

# ── 2. 切批（每 10 条一批）───────────────────────────────────────────────────
BATCH_SIZE = 10
batches = [valid_items[i : i + BATCH_SIZE] for i in range(0, len(valid_items), BATCH_SIZE)]
print(f"共 {len(batches)} 个批次，每批最多 {BATCH_SIZE} 条")

# ── 3. 辅助函数 ───────────────────────────────────────────────────────────────
COMPRESSION_OUTPUT = "./kh_compression_zqrz317_fjrjg.json"
compression_lock = Lock()


def format_batch_snippets(batch):
    """将一批 Know_How 格式化为 compression_v0 所需的输入文本。"""
    parts = []
    for i, item in enumerate(batch, 1):
        parts.append(f"【片段 {i}】（原始 index: {item['index']}）\n{item['Know_How']}")
    return "\n\n---\n\n".join(parts)


def _update_json_file_compression(file_path, key, value):
    """线程安全地更新 JSON 文件（读 → 改 → 写）。"""
    data_dict = {}
    if os.path.exists(file_path):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data_dict = json.load(f)
        except (json.JSONDecodeError, IOError):
            data_dict = {}
    data_dict[key] = value
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data_dict, f, ensure_ascii=False, indent=2)


# ── 4. 单批次处理（带重试）────────────────────────────────────────────────────
def process_compression_batch(batch_idx, batch, output_file, max_retries=999):
    """对单个批次调用 compression_v0 做知识整合，失败时指数退避重试。"""
    retry_count = 0
    base_wait = 3
    last_error_msg = ""

    while True:
        if retry_count > max_retries:
            error_result = {
                "batch_index": batch_idx,
                "source_indices": [item["index"] for item in batch],
                "item_count": len(batch),
                "status": "failed",
                "error": last_error_msg,
                "retry_count": retry_count,
            }
            with compression_lock:
                _update_json_file_compression(output_file, str(batch_idx), error_result)
            print(f"[Failed] 批次 {batch_idx} 超过最大重试次数 ({max_retries})")
            return batch_idx, "failed", None

        try:
            if retry_count > 0:
                print(f"[Retry {retry_count}] 批次 {batch_idx} 第 {retry_count + 1} 次尝试...")

            snippets_text = format_batch_snippets(batch)
            response = chat(compression_v2(f=snippets_text))

            try:
                content = safe_parse_json(response["content"])
            except Exception as json_err:
                raise Exception(
                    f"JSON 解析失败（含兜底策略）: {json_err}"
                )

            if "Final_Know_How" not in content:
                raise Exception("响应缺少必需字段 'Final_Know_How'")

            result = {
                "batch_index": batch_idx,
                "source_indices": [item["index"] for item in batch],
                "item_count": len(batch),
                "Synthesis_Summary": content.get("Synthesis_Summary", ""),
                "Final_Know_How": content["Final_Know_How"],
                "status": "success",
                "retry_count": retry_count,
            }

            with compression_lock:
                _update_json_file_compression(output_file, str(batch_idx), result)

            suffix = f"（历经 {retry_count} 次重试）" if retry_count > 0 else ""
            print(f"[Success] 批次 {batch_idx + 1}/{len(batches)} 完成（{len(batch)} 条知识）{suffix}")
            return batch_idx, "success", result

        except Exception as e:
            retry_count += 1
            last_error_msg = str(e)
            wait_time = 3# min(base_wait * (2 ** (retry_count - 1)), 60)
            print(f"[Error] 批次 {batch_idx} 第 {retry_count} 次失败: {last_error_msg[:150]}")
            print(f"[Wait] 等待 {wait_time} 秒后重试...")
            time.sleep(wait_time)


# ── 5. 多线程批量整合（支持断点续传）─────────────────────────────────────────
def run_compression(batches, max_workers=3, output_file=COMPRESSION_OUTPUT):
    existing_data = {}
    if os.path.exists(output_file):
        try:
            with open(output_file, "r", encoding="utf-8") as f:
                existing_data = json.load(f)
            print(f"📂 发现已有进度文件，包含 {len(existing_data)} 个批次记录，自动续传")
        except Exception:
            pass

    pending = [
        (i, b)
        for i, b in enumerate(batches)
        if str(i) not in existing_data
        or existing_data.get(str(i), {}).get("status") != "success"
    ]

    completed = len(batches) - len(pending)
    print(f"总批次: {len(batches)}，已完成: {completed}，待处理: {len(pending)}，并发数: {max_workers}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_compression_batch, i, b, output_file): i
            for i, b in pending
        }

        for future in as_completed(future_to_idx):
            batch_idx = future_to_idx[future]
            try:
                _, status, _ = future.result()
                if status == "success":
                    completed += 1
                    print(f"✅ 进度: {completed}/{len(batches)} ({completed / len(batches) * 100:.1f}%)")
                else:
                    print(f"❌ 批次 {batch_idx} 最终失败")
            except Exception as e:
                print(f"💥 批次 {batch_idx} 处理异常: {e}")

    print(f"\n🏁 全部完成！结果保存于: {output_file}")


# ── 6. 启动 ───────────────────────────────────────────────────────────────────
run_compression(batches, max_workers=16)

有效 Know_How 总数: 151
共 16 个批次，每批最多 10 条
📂 发现已有进度文件，包含 16 个批次记录，自动续传
总批次: 16，已完成: 16，待处理: 0，并发数: 16

🏁 全部完成！结果保存于: ./kh_compression_zqrz317_fjrjg.json


# 三级提炼：对二级提炼后的整合know-how，做逻辑合并、去重（若已有业务范围框定，则按业务范围独立整合）

# 表格拼接

In [16]:
import json
with open("kh_results_zqrz317_fjrjg.json", "r", encoding="utf-8") as f:
    level1 = json.load(f)

In [17]:
import json
import openpyxl
from openpyxl.styles import Alignment

# ── 1. 写入一级 KH（每行独立提炼结果） ───────────────────────────────────────
for i in range(len(level1)):
    l1_data = level1[str(i)]
    know_how = l1_data['Know_How']
    logic = l1_data['Logic_Diagnosis']
    data.loc[i, "know-how"] = know_how
    data.loc[i, "Logic_Diagnosis"] = logic

# ── 2. 加载二级（batch 压缩）和三级（最终合并）数据 ──────────────────────────
with open("kh_compression_zqrz317_experts.json", "r", encoding="utf-8") as f:
    level2 = json.load(f)

# with open("kh_merge_progress_zqrz317_experts.json", "r", encoding="utf-8") as f:
#     level3_data = json.load(f)
# level3_kh = level3_data["current_kh"]

# ── 3. 构建 原始行索引 → 二级批次 key 的映射 ─────────────────────────────────
row_to_l2_key = {}
for k, v in level2.items():
    for row_idx in v["source_indices"]:
        row_to_l2_key[row_idx] = k

# ── 4. 向 data 写入二级和三级 KH 列 ──────────────────────────────────────────
data["二级提炼KH"] = None
# data["三级提炼KH"] = level3_kh  # 所有行相同

for row_idx in data.index:
    l2_key = row_to_l2_key.get(row_idx)
    if l2_key is not None:
        data.at[row_idx, "二级提炼KH"] = level2[l2_key]["Final_Know_How"]

print(f"二级 KH 覆盖行数: {data['二级提炼KH'].notna().sum()} / {len(data)}")

# ── 5. 导出 Excel（pandas 初步写入） ─────────────────────────────────────────
output_path = "output_kh_zqrz317_fjrjg.xlsx"
data.to_excel(output_path, index=False, engine="openpyxl")

# ── 6. 重新加载，进行合并单元格处理 ──────────────────────────────────────────
wb = openpyxl.load_workbook(output_path)
ws = wb.active

# 获取各 KH 列的列号（1-based）
header = [cell.value for cell in ws[1]]
col_l2 = header.index("二级提炼KH") + 1
# col_l3 = header.index("三级提炼KH") + 1

# 构建 data 行标签 → Excel 行号（Excel 行 2 对应 data 第 0 个元素，行 1 是表头）
index_list = list(data.index)
label_to_xl = {label: pos + 2 for pos, label in enumerate(index_list)}

# ── 6a. 合并二级 KH 列：同一 level2 batch 的行合并为一个单元格 ────────────────
l2_batch_xl_rows = {}
for label, xl_row in label_to_xl.items():
    l2_key = row_to_l2_key.get(label)
    if l2_key is not None:
        l2_batch_xl_rows.setdefault(l2_key, []).append(xl_row)

for l2_key, xl_rows in sorted(l2_batch_xl_rows.items()):
    if len(xl_rows) <= 1:
        continue
    xl_rows.sort()
    ws.merge_cells(
        start_row=xl_rows[0], start_column=col_l2,
        end_row=xl_rows[-1], end_column=col_l2
    )
    ws.cell(row=xl_rows[0], column=col_l2).alignment = Alignment(
        wrap_text=True, vertical="top"
    )

# ── 6b. 合并三级 KH 列：全部数据行合并为一个单元格 ───────────────────────────
# n = len(data)
# if n > 1:
#     ws.merge_cells(start_row=2, start_column=col_l3, end_row=n + 1, end_column=col_l3)
#     ws.cell(row=2, column=col_l3).alignment = Alignment(wrap_text=True, vertical="top")

wb.save(output_path)
print(f"✅ 已保存至 {output_path}（共 {n} 行数据，二级批次 {len(l2_batch_xl_rows)} 组合并）")

二级 KH 覆盖行数: 148 / 152


NameError: name 'n' is not defined

# 推理测试

In [15]:
test_cut = int(cut*0.7)
data_test = data.loc[test_cut:,:]
data_test.reset_index(drop = True, inplace = True)
data_test[['question','answer','confirmed']].to_csv("./fjrjg_test_data.csv")

In [16]:
import json
import time
import pandas as pd
from openai import OpenAI
import concurrent.futures
from tqdm import tqdm  # 用于显示进度条

INPUT_JSON_PATH = "kh_compression_zqrz317_fjrjg.json"
INPUT_CSV_PATH = "fjrjg_test_data.csv"
OUTPUT_CSV_PATH = "./kh_mapreduce_result_0318.csv"

# 线程池最大工作线程数（相当于并发数，防止触发大模型 API 的 Rate Limit）
MAX_WORKERS = 16 

def clean_json_string(text: str) -> str:
    """清理大模型可能输出的 Markdown 标记，确保能被 json.loads 解析"""
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

# --- Map 阶段的线程任务 ---
def run_map_task(task_input: dict) -> dict:
    q_idx = task_input["q_idx"]
    question = task_input["question"]
    kh_id = task_input["kh_id"]
    kh_text = task_input["kh_text"]
    pp = potential_pitfalls()
    prompt = infer_v1(question, kh_text, pp)
    
    try:
        # 增加异常容错，防止因单个请求 JSON 报错中断整个线程池
        raw_content = chat(prompt)['content']
        result = json.loads(clean_json_string(raw_content))
    except Exception as e:
        result = {
            "Match_Status": "NO",
            "Rejection_Reason": f"模型调用或JSON解析失败: {str(e)}",
            "Reasoning_Chain": "",
            "Derived_Answer": ""
        }
    
    # 将标识信息塞回结果中，方便后续 Reduce 组装
    result["q_idx"] = q_idx
    result["kh_source_id"] = kh_id
    return result

# --- Reduce 阶段的线程任务 ---
def run_reduce_task(task_input: dict) -> dict:
    q_idx = task_input["q_idx"]
    question = task_input["question"]
    valid_results = task_input["valid_results"]
    
    # ----------------------------------------------------
    # 新增逻辑：1. 让大模型先“裸考”问题，获取 Extra_Information
    # ----------------------------------------------------
    extra_prompt = f"作为一个资深行业顾问，请结合你的专业知识，直接且详尽地解答以下问题：\n【用户问题】：{question}"
    try:
        extra_info = chat(extra_prompt, vendor = "volc", model = "deepseek-v3.2")['content']
    except Exception as e:
        extra_info = f"获取额外信息失败: {str(e)}"
    
    # 2. 组装 Valid_Candidate_Answers
    if not valid_results:
        candidates_text = "【空】所有并发线程均拒答，未匹配到有效KNOW-HOW。"
    else:
        # 将所有成功的推导答案拼接
        candidates_text = "\n\n".join(
            [f"--- 来自知识块 [{r.get('kh_source_id')}] 的推理 ---\n"
             f"推导逻辑: {r.get('Reasoning_Chain')}\n"
             f"候选答案: {r.get('Derived_Answer')}" 
             for r in valid_results]
        )
    
    # 3. 将三个变量传入您修改后的 summary_v0 
    prompt = summary_v0(question, extra_info, candidates_text)
    
    try:
        raw_content = chat(prompt)['content']
        reduce_result = json.loads(clean_json_string(raw_content))
    except Exception as e:
        reduce_result = {
            "Synthesis_Analysis": f"解析失败: {str(e)}",
            "Final_Answer": raw_content if 'raw_content' in locals() else "处理失败"
        }
        
    reduce_result["q_idx"] = q_idx
    # ★ 将 Extra_Information 塞入返回字典，以便写入 DataFrame
    reduce_result["Extra_Information"] = extra_info 
    
    return reduce_result


def main():
    # 1. 加载数据
    with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        kh_data = json.load(f)
    print(f"[*] 成功加载 {len(kh_data)} 条全局行业 KNOW-HOW。")

    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"[*] 成功加载 {len(df)} 条测试问题。")
    
    # 2. 准备全局 Map 任务池 (扁平化拆解：问题数 x KNOW-HOW数)
    map_task_inputs = []
    for index, row in df.iterrows():
        question = str(row.get('question', ''))
        if not question.strip():
            continue
        for kh_id, kh_info in kh_data.items():
            kh_text = kh_info.get("Final_Know_How", "")
            map_task_inputs.append({
                "q_idx": index,
                "question": question,
                "kh_id": kh_id,
                "kh_text": kh_text
            })
            
    print(f"\n🚀 [阶段一] 开始并发 MAP 任务... (共计 {len(map_task_inputs)} 个子任务)")
    map_results = []
    
    # 启动线程池执行 MAP
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # tqdm 结合 executor.map 制作进度条
        futures = {executor.submit(run_map_task, task): task for task in map_task_inputs}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Map Progress"):
            try:
                map_results.append(future.result())
            except Exception as exc:
                print(f"[-] Map 任务引发了异常: {exc}")

    # 3. 按问题 ID (q_idx) 聚合并梳理 Map 结果
    grouped_map_results = {i: [] for i in df.index}
    for res in map_results:
        q_idx = res.get("q_idx")
        if q_idx is not None:
            grouped_map_results[q_idx].append(res)
            
    # 准备全局 Reduce 任务池
    reduce_task_inputs = []
    for q_idx, results in grouped_map_results.items():
        question = str(df.at[q_idx, 'question'])
        # 筛选出 Match_Status 为 YES 的有效结果
        valid_results = [r for r in results if r.get("Match_Status") == "YES"]
        
        reduce_task_inputs.append({
            "q_idx": q_idx,
            "question": question,
            "valid_results": valid_results,
            "all_map_results": results  # 保存全量日志备用
        })

    print(f"\n🚀 [阶段二] 开始并发 REDUCE 任务 (包含抽取 Extra Info)... (共计 {len(reduce_task_inputs)} 个子任务)")
    reduce_results = {}
    
    # 启动线程池执行 REDUCE
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(run_reduce_task, task): task for task in reduce_task_inputs}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Reduce Progress"):
            try:
                res = future.result()
                reduce_results[res["q_idx"]] = res
            except Exception as exc:
                print(f"[-] Reduce 任务引发了异常: {exc}")

    # 4. 数据回写至 DataFrame
    print("\n📝 正在将中间步骤和最终推理结果回写到 CSV...")
    
    # --- 新增 DataFrame 列 ---
    df['Extra_Information'] = ""
    df['Map_Total_Evaluated'] = 0
    df['Map_Match_Count'] = 0
    df['Map_Valid_Details'] = ""
    df['Map_Rejected_Reasons'] = ""
    df['Reduce_Analysis'] = ""
    df['Final_Inference_Answer'] = ""

    for task_info in reduce_task_inputs:
        q_idx = task_info["q_idx"]
        all_maps = task_info["all_map_results"]
        valids = task_info["valid_results"]
        
        # 统计
        rejected_reasons = [f"KH[{r.get('kh_source_id')}]: {r.get('Rejection_Reason')}" for r in all_maps if r.get("Match_Status") == "NO"]
        
        # Reduce 结果
        r_res = reduce_results.get(q_idx, {})
        
        # 赋值回写
        df.at[q_idx, 'Extra_Information'] = r_res.get("Extra_Information", "")
        df.at[q_idx, 'Map_Total_Evaluated'] = len(all_maps)
        df.at[q_idx, 'Map_Match_Count'] = len(valids)
        df.at[q_idx, 'Map_Valid_Details'] = json.dumps(valids, ensure_ascii=False) if valids else "None"
        df.at[q_idx, 'Map_Rejected_Reasons'] = "\n".join(rejected_reasons)
        df.at[q_idx, 'Reduce_Analysis'] = r_res.get("Synthesis_Analysis", "")
        df.at[q_idx, 'Final_Inference_Answer'] = r_res.get("Final_Answer", "")

    # 5. 保存文件
    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
    print(f"\n🎉 所有推理任务圆满完成！带 Extra_Info 的结果已保存至: {OUTPUT_CSV_PATH}")

    
main()

[*] 成功加载 16 条全局行业 KNOW-HOW。
[*] 成功加载 46 条测试问题。

🚀 [阶段一] 开始并发 MAP 任务... (共计 736 个子任务)


Map Progress: 100%|██████████| 736/736 [05:01<00:00,  2.44it/s]



🚀 [阶段二] 开始并发 REDUCE 任务 (包含抽取 Extra Info)... (共计 46 个子任务)


Reduce Progress: 100%|██████████| 46/46 [03:17<00:00,  4.30s/it]


📝 正在将中间步骤和最终推理结果回写到 CSV...

🎉 所有推理任务圆满完成！带 Extra_Info 的结果已保存至: ./kh_mapreduce_result_0318.csv
